# AI training

This AI is based on the `Sklearn` Ensemble learning method that is trained on the dataset.

In [3]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append(os.path.abspath(".."))
from src.airfoil_predictor import AirfoilAI
import numpy as np
import plotly.graph_objects as go

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## AI model

This AI model is trained and saved under the models/ folder

In [5]:
ai = AirfoilAI()
ai.train("../data/airfoil_optimization_results.csv")
ai.save_model()

✅ AI finished training with standardized features.
Saved: ../models\airfoil_model.pkl and ../models\scaler.pkl


Once the AI is trained it can be load directly from the models/ folder

In [6]:
ai2 = AirfoilAI(model_path="../models/airfoil_model.pkl", 
               scaler_path="../models/scaler.pkl")

AI Loaded from ../models


In [7]:
def plot_ai_prediction_surface(ai_model, data="finesse_max", size=800, resolution=300):
    """
    Dataset visualization function for airfoil AI results.
    """

    mapping = {
        "best_m": 0,
        "best_p": 1,
        "best_t": 2,
        "finesse_max": 3
    }

    param_idx = mapping[data]
    
    v_range = np.linspace(50, 250, resolution)
    alt_range = np.linspace(0, 12000, resolution)
    V, Alt = np.meshgrid(v_range, alt_range)
    
    grid_points = np.c_[V.ravel(), Alt.ravel()]
    grid_scaled = ai_model.scaler.transform(grid_points)
    predictions = ai_model.model.predict(grid_scaled)
    

    Z = predictions[:, param_idx].reshape(V.shape)

    cmap = "Viridis"
    
    fig = go.Figure(data=go.Heatmap(
        z=Z,
        x=v_range,
        y=alt_range,
        colorscale=cmap,
        colorbar=dict(title=data),
        hovertemplate="Vitesse: %{x:.1f} m/s<br>Altitude: %{y:.0f} m<br>Prédit: %{z:.2f}<extra></extra>"
    ))
    
    fig.update_layout(
        title=f"IA Surrogate Model : {data}",
        xaxis_title="Velocity (m/s)",
        yaxis_title="Altitude (m)",
        width=size,
        height=size,
        title_x=0.5,
        template="plotly_white"
    )
    
    return fig

In [8]:
fig_m = plot_ai_prediction_surface(ai2, data="best_m", size=1000, resolution=500)

fig_m.show()

c:\Users\bertr\Documents\Code\Foil-Optimization-AI\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
